In [0]:
# Databricks notebook source
# DATA DRIFT CHECK (KS test)
# Question: is the new data (holdout) still similar to the data the model was trained on (train)?
# Output: one table that the dashboard reads
#   data_drift_results -> one row per feature (KS statistic, p-value, drift level)

# COMMAND ----------

import numpy as np
from datetime import datetime
from pyspark.sql import SparkSession
from scipy.stats import ks_2samp

spark = SparkSession.builder.getOrCreate()

# COMMAND ----------

# SETTINGS - change only this block
REFERENCE_TABLE = "catalog_mle.ml.demand_features_train"            # what the model learned from
CURRENT_TABLE = "catalog_mle.ml.demand_features_inference_holdout"  # new data
RESULTS_TABLE = "catalog_mle.ml.data_drift_results"

FEATURES = ["avg_unit_price", "total_inventory", "day_of_week", "month",
            "demand_lag_1", "demand_lag_7", "rolling_avg_7"]

P_LIMIT = 0.05    # p-value below this  = the two datasets are different
BIG_GAP = 0.1     # KS statistic above this = the difference is big (rule of thumb)

# COMMAND ----------


def get_values(table, feature):
    """Read one column, drop empty values, sort it small to big."""
    values = spark.table(table).select(feature).toPandas()[feature].dropna()
    return np.sort(values.astype(float).values)


def drift_level(ks_stat, p_value):
    """Turn the two KS numbers into a simple label."""
    if p_value >= P_LIMIT:
        return "Stable", "No meaningful change"
    if ks_stat < BIG_GAP:
        return "Minor drift (watch)", "Different, but the gap is small"
    return "Major drift", "The data has clearly changed"


# COMMAND ----------

# Run the KS test for every feature
checked_at = datetime.now()
result_rows = []

for feature in FEATURES:
    ref = get_values(REFERENCE_TABLE, feature)   # Training values
    cur = get_values(CURRENT_TABLE, feature)     # Current values

    # KS test: statistic = biggest gap between the two lines (0 to 1), p_value = chance it is just luck
    ks_stat, p_value = ks_2samp(ref, cur)
    level, meaning = drift_level(ks_stat, p_value)

    result_rows.append((feature, float(ks_stat), float(p_value), level, meaning, checked_at))

# COMMAND ----------

# Save the table (overwrite = always keep the latest check)
columns = ["feature", "ks_statistic", "p_value", "drift_level", "meaning", "checked_at"]

(spark.createDataFrame(result_rows, columns).write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(RESULTS_TABLE))

print(f"Saved {len(result_rows)} rows -> {RESULTS_TABLE}")